<a href="https://colab.research.google.com/github/ragiokay/AI_transcribe/blob/main/AI_transcribe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install faster-whisper google-generativeai python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 60.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 96.8 MB/s eta 0:00:00


In [3]:
import os
import time
from google.colab import drive
import google.generativeai as genai
from google.colab import userdata
from faster_whisper import WhisperModel

# ==========================================
# 1. 自動掛載 Google Drive 與路徑設定
# ==========================================
print("[開始] 正在掛載 Google Drive...")
drive.mount('/content/drive')

# 設定你在 My Drive 下建立的資料夾名稱
DRIVE_DIR = "/content/drive/MyDrive/AI_transcribe"
os.makedirs(DRIVE_DIR, exist_ok=True)

# ─── 📝 請在這裡修改你的輸入音檔名稱 ───
AUDIO_FILE_NAME = "NLP.m4a"  # 請把這個音檔直接上傳到雲端硬碟的 AI_transcribe 資料夾中
# ──────────────────────────────────────

AUDIO_FILE_PATH = os.path.join(DRIVE_DIR, AUDIO_FILE_NAME)

# ==========================================
# 2. 安全讀取 Gemini API Key 設定
# ==========================================
try:
    # 從 Colab 左側的 Secrets 讀取密鑰，避免寫死在代碼中
    gemini_key = userdata.get('GEMINI_API_KEY')
    genai.configure(api_key=gemini_key)
except Exception:
    # 如果沒設 Secrets，也可以手動貼在這裡當備用
    # genai.configure(api_key="你的_GEMINI_API_KEY")
    print("⚠️ 未偵測到 Colab Secrets 中的 GEMINI_API_KEY，請確保已手動設定。")

# 使用適合長文本與技術內容修復的 Gemini 1.5 Pro
LLM_MODEL_NAME = "gemini-1.5-pro"

# ==========================================
# 3. Whisper A100 最高速配置
# ==========================================
WHISPER_MODEL_SIZE = "large-v3"
DEVICE = "cuda"            # 偵測並使用 GPU
COMPUTE_TYPE = "float16"   # A100 啟用半精度加速，速度加倍且省記憶體

# ==========================================
# 4. 核心功能實作
# ==========================================

def run_transcription(file_path):
    """階段一：使用 faster-whisper 進行極速語音辨識"""
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"在雲端硬碟資料夾中找不到音檔：{file_path}\n請確認已將檔案上傳至 Google Drive 的 AI_transcribe 資料夾中。")

    print(f"\n🚀 [1/3] 正在本地 A100 加速載入 Whisper {WHISPER_MODEL_SIZE} 模型...")
    # 模型下載到本地 /root/.cache，確保運作時的最高讀寫速度
    model = WhisperModel(WHISPER_MODEL_SIZE, device=DEVICE, compute_type=COMPUTE_TYPE)

    print(f"🎙️ [2/3] 開始語音辨識音檔: {os.path.basename(file_path)}")
    start_time = time.time()

    # beam_size=5 平衡速度與精準度；可加上 language="zh" 稍微加快強制辨識中文
    segments, info = model.transcribe(file_path, beam_size=5, language="zh")
    print(f"📊 偵測到主要語言: {info.language} (信心度: {info.language_probability:.2f})")

    raw_text = ""
    for segment in segments:
        # 即時在終端機列印進度，讓你知道它還活著
        print(f"[{segment.start:.1f}s -> {segment.end:.1f}s] {segment.text}")
        raw_text += segment.text + " "

    elapsed_time = time.time() - start_time
    print(f"✨ Whisper 辨識完成！總耗時: {elapsed_time:.2f} 秒")
    return raw_text.strip()

def run_llm_fix(raw_text):
    """階段二：使用 Gemini 1.5 Pro 進行上下文語意修復與結構化"""
    print(f"\n🧠 [3/3] 正在呼叫 {LLM_MODEL_NAME} 進行上下文語意修復與去口語化...")

    system_prompt = """
    你是一位頂尖的自然語言處理 (NLP) 與人工智慧領域的學術編輯。
    我會給你一份資工所學生與助教方針對「NLP 期末專案 (Group 5)」的討論會議原始逐字稿（包含大量口頭禪與語音辨識錯誤）。
    該專案的主題為探討「情緒語氣是否會干擾 LLM 的邏輯判斷」，並使用了「原子事實 (Atomic Facts) 拆解」等評估手法。

    請幫我執行以下任務：
    1. 語句修復：根據上下文邏輯，修復破碎句型、補上正確的標點符號，刪除無意義的「呃」、「然後」、「就是說」、「對」等口語贅詞。
    2. 結構化排版：依照討論的邏輯自動分段，並加上清晰的「小標題 (Markdown ##)」，例如：Slide 修改建議、實驗設計討論、助教回饋、下一步行動 (Action Items) 等。
    3. 保留專業術語：**絕對要保留 NLP 領域的專業術語與中英夾雜習慣**（例如：LLM, Prompt Injection, Atomic Facts, baseline, p-value, few-shot, hallucination, true/fake/neutral, slide, proposal 等），不要強制翻譯成中文。
    4. 數據精準度：會議中討論到的任何具體實驗數據（如 23 題、p < 0.001 等）必須百分之百精準保留，不可竄改。
    5. 忠於原意：精準修復講者對話，絕對不要憑空捏造或延伸講者與助教沒有說過的內容。

    請直接輸出修復並排版後的完整 Markdown 會議紀錄。
    """

    model = genai.GenerativeModel(
        model_name=LLM_MODEL_NAME,
        system_instruction=system_prompt
    )

    # 溫度設低 (0.2) 確保極度忠於原文，不胡亂發揮
    generation_config = genai.GenerationConfig(temperature=0.2)

    start_time = time.time()
    response = model.generate_content(raw_text, generation_config=generation_config)
    elapsed_time = time.time() - start_time

    print(f"✨ Gemini 語意修復完成！總耗時: {elapsed_time:.2f} 秒")
    return response.text

# ==========================================
# 5. 主流程控制
# ==========================================
if __name__ == "__main__":
    base_name = os.path.splitext(AUDIO_FILE_NAME)[0]

    # 定義輸出的雲端硬碟路徑
    raw_output_path = os.path.join(DRIVE_DIR, f"{base_name}_原始逐字稿.txt")
    fixed_output_path = os.path.join(DRIVE_DIR, f"{base_name}_修復完成版.md")

    try:
        # 1. 跑轉錄
        raw_transcript = run_transcription(AUDIO_FILE_PATH)

        # 轉錄完立刻寫入雲端硬碟（買個保險）
        with open(raw_output_path, "w", encoding="utf-8") as f:
            f.write(raw_transcript)
        print(f"💾 原始逐字稿已即時備份至 Drive: {raw_output_path}")

        # 2. 跑 LLM 修復
        fixed_transcript = run_llm_fix(raw_transcript)

        # 最終結果寫入雲端硬碟 (使用 .md 格式，方便你之後在 Obsidian 或 GitHub 直接看精美排版)
        with open(fixed_output_path, "w", encoding="utf-8") as f:
            f.write(fixed_transcript)
        print(f"💾 終端修復版已完美儲存至 Drive: {fixed_output_path}")

        print(f"\n🎉 恭喜！所有任務安全完成。請至 Google Drive 的 AI_transcribe 資料夾查看結果。")

    except Exception as e:
        print(f"\n❌ 程式執行中途發生錯誤: {e}")

[開始] 正在掛載 Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

🚀 [1/3] 正在本地 A100 加速載入 Whisper large-v3 模型...
🎙️ [2/3] 開始語音辨識音檔: NLP.m4a
📊 偵測到主要語言: zh (信心度: 1.00)
[0.0s -> 5.0s] 我們會先講一下我們的實驗流程
[5.0s -> 8.0s] 我們是用Cofed這個資料集
[8.0s -> 11.0s] 然後我們會給大家做一些資料的過濾
[11.0s -> 14.0s] 就是它如果這個資料集太長太短會掉
[14.0s -> 16.0s] 然後會取得原始資料
[16.0s -> 19.0s] 然後我們會用LLM來把它做改寫
[19.0s -> 22.0s] 我們是要做改寫語氣之後的判斷
[22.0s -> 24.0s] 就是這個改寫語氣之後
[24.0s -> 26.0s] 會不會影響到LLM的判斷
[26.0s -> 29.0s] 所以我們會先把這邊得到原始訊息
[29.0s -> 32.0s] 然後再給大家做一個過濾的判斷
[32.0s -> 34.0s] 然後之後用情緒分析
[34.0s -> 36.0s] 好這是資料集
[36.0s -> 38.0s] 然後這次要解內容
[38.0s -> 40.0s] 這會不會有原始的訊息
[40.0s -> 43.0s] 會有原始的訊息
[43.0s -> 45.0s] 我想要事實查核
[45.0s -> 48.0s] 那你們跟情緒的關係是怎樣
[48.0s -> 51.0s] 我們會後面會提到
[51.0s -> 53.0s] 所以我們要改寫
[53.0s -> 54.0s] 就是我們要加入情緒
[54.0s -> 58.0s] 我想到你們是說要把原本有一個資料集
[58.0s -> 59.0s] 要把它加情緒
[59.0s -> 60.0s] 對然後看它
[60.0s -> 61.0s] 有沒有判斷
[61.0s -> 62.0s] 或會不會跟原始
[62.0s -> 


❌ 程式執行中途發生錯誤: 404 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-pro:generateContent?%24alt=json%3Benum-encoding%3Dint: models/gemini-1.5-pro is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.
